# LDM training and Hugging Face release

Run the cells in order for one variant at a time. Training checkpoints are saved to Google Drive; prepared arrays and snapshots are rebuilt in Colab as needed. The upload cell is disabled by default. Use a GPU runtime for production training when available. Python 3.10 or newer is supported by the project.

This notebook uses the private GitHub repository `appodoom/ldm`, the pinned `Appodoom/ldm-data` dataset, and the existing `Appodoom/ldm-with-subd` / `Appodoom/ldm-no-subd` model repositories. A successful local snapshot check does not upload anything.

## 1. Clone or update the code

Add `GITHUB_TOKEN` to Colab Secrets, grant this notebook access, and use a token with read access to the private repository. The token is passed to Git only for this command and is not written into the repository URL.

In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

project = Path("/content/ldm")
token = userdata.get("GITHUB_TOKEN")
assert token, "Add GITHUB_TOKEN to Colab Secrets and allow notebook access"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
})
if (project / ".git").exists():
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", "main"],
                   env=git_env, check=True)
elif project.exists():
    raise RuntimeError("/content/ldm exists but is not the Git checkout")
else:
    subprocess.run(["git", "clone", "https://github.com/appodoom/ldm.git", str(project)],
                   env=git_env, check=True)
code_commit = subprocess.check_output(
    ["git", "-C", str(project), "rev-parse", "HEAD"], text=True).strip()
print("Code commit:", code_commit)


## 2. Install and run the package checks

In [ ]:
os.chdir(project)
print("Python:", sys.version.split()[0])
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               check=True)


## 3. Sign in to Hugging Face and load the pinned dataset

The Hugging Face account must be able to read the private dataset. To upload later, it must also have write access to the selected model repository.

In [ ]:
from huggingface_hub import notebook_login, snapshot_download

notebook_login()
data_root = Path(snapshot_download(
    repo_id="Appodoom/ldm-data", repo_type="dataset",
    revision="34fb2449a25472e856a4db0aa85e280e9a82a8b2",
    allow_patterns="five-line/**", local_dir=project / ".derbake",
))
ordinary = data_root / "five-line" / "normal"
decorated = data_root / "five-line" / "decorated"
assert len(list(ordinary.glob("*.derbake"))) == 116
assert len(list(decorated.glob("*.derbake"))) == 116
print("Paired files:", len(list(ordinary.glob("*.derbake"))))


## 4. Prepare one variant

Change `variant` to `no_subd` for the second model. Each paired file is split by complete cycles into approximately 70/20/10 train/validation/test ranges before chunks are made.

In [ ]:
import json

variant = "with_subd"  # Change to "no_subd" for the other model.
assert variant in {"with_subd", "no_subd"}
dataset = project / "datasets" / variant
manifest_path = dataset / "dataset_manifest.json"
if not manifest_path.exists():
    subprocess.run([
        sys.executable, "-m", "ldm.cli", "prepare",
        "--ordinary", str(ordinary), "--decorated", str(decorated),
        "--variant", variant, "--output", str(dataset),
    ], check=True)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["variant"] == variant
print("Prepared dataset:", dataset)
print("Split chunks:", {name: manifest["stats"][name]["chunks"]
                        for name in ("train", "val", "test")})


## 5. Mount persistent storage

Colab's `/content` can disappear when the runtime ends. Checkpoints and release records are therefore saved under `MyDrive/LDM`. In a new runtime, rerun the earlier cells to recreate the same pinned dataset, then resume from Drive.

In [ ]:
from google.colab import drive
import torch

drive.mount("/content/drive")
storage_root = Path("/content/drive/MyDrive/LDM")
run = storage_root / "runs" / variant
run.parent.mkdir(parents=True, exist_ok=True)
print("Run folder:", run)
print("PyTorch CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Training will use CPU. Select a GPU runtime for a production run if available.")


## 6. Train or resume

Set the total target number of epochs below. `last.pt` resumes an interrupted run; `best.pt` is the checkpoint selected by validation loss. Keep the model settings and prepared dataset identical when resuming. Running this cell after the requested epochs are complete skips training. During training, Colab prints average training loss every 50 optimizer steps, checkpoint saves, and validation loss at each epoch end.

In [ ]:
target_epochs = 10  # Total epochs, including completed epochs when resuming.
assert target_epochs >= 1
metrics_path = run / "metrics.json"
completed_epochs = 0
if metrics_path.exists():
    previous_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    completed_epochs = previous_metrics["history"][-1]["epoch"]

if completed_epochs >= target_epochs:
    print(f"Already completed {completed_epochs} epochs; training skipped.")
else:
    command = [sys.executable, "-u", "-m", "ldm.cli", "train",
               "--dataset", str(dataset), "--run", str(run),
               "--epochs", str(target_epochs),
               "--log-every-steps", "50"]
    if (run / "last.pt").exists():
        command.append("--resume")
    elif run.exists() and any(run.iterdir()):
        raise RuntimeError("Run folder has no resumable last.pt. Inspect it before choosing a new run folder.")
    subprocess.run(command, check=True)


## 7. Review training results

Losses show model fit; they do not by themselves establish rhythm quality. Inspect generated examples below before deciding to upload.

In [ ]:
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps({
    "completed_epochs": metrics["history"][-1]["epoch"],
    "steps": metrics["steps"],
    "best_val_loss": metrics["best_val_loss"],
    "test_loss": metrics["test_loss"],
    "history": metrics["history"],
}, indent=2))


## 8. Build and validate a local Hugging Face snapshot

The folder name includes the selected checkpoint's hash, so rerunning this cell reuses only the matching snapshot. Validation loads the model, tokenizer, and processor in an isolated process and checks a forward pass and `.derbake` round trip.

In [ ]:
import hashlib

best_checkpoint = run / "best.pt"
assert best_checkpoint.is_file(), "Training has not saved best.pt"
checkpoint_sha256 = hashlib.sha256(best_checkpoint.read_bytes()).hexdigest()
snapshot = project / "snapshots" / f"{variant}-{checkpoint_sha256[:12]}-{code_commit[:8]}"
release_manifest_path = snapshot / "metadata" / "release_manifest.json"
if snapshot.exists() and not release_manifest_path.exists():
    raise RuntimeError(f"Incomplete snapshot at {snapshot}; choose a new empty folder after inspecting it")
if not snapshot.exists():
    subprocess.run([sys.executable, "-m", "ldm.cli", "snapshot",
                    "--run", str(run), "--output", str(snapshot)], check=True)
release_manifest = json.loads(release_manifest_path.read_text(encoding="utf-8"))
assert release_manifest["source_checkpoint_sha256"] == checkpoint_sha256
assert release_manifest["dataset_variant"] == variant
assert release_manifest["dataset_fingerprint"] == metrics["dataset_fingerprint"]
source_code_path = snapshot / "metadata" / "source_code.json"
if source_code_path.exists():
    assert json.loads(source_code_path.read_text(encoding="utf-8"))["git_commit"] == code_commit
else:
    source_code_path.write_text(json.dumps({"git_commit": code_commit}, indent=2) + "\n",
                                encoding="utf-8")
subprocess.run([sys.executable, "-m", "ldm.cli", "validate",
                "--snapshot", str(snapshot)], check=True)
print("Validated snapshot:", snapshot)


## 9. Generate and inspect a rhythm

This is a practical quality check beyond the forward-pass smoke test. It uses one complete cycle from the pinned corpus as a prompt and asks for one new cycle. Inspect the printed five-line file and generation report. A failed generation is a reason to investigate before release.

In [ ]:
from transformers import AutoModel, AutoProcessor
from ldm.derbake_ldm import RhythmSequence, parse_derbake

processor = AutoProcessor.from_pretrained(snapshot, trust_remote_code=True)
model = AutoModel.from_pretrained(snapshot, trust_remote_code=True).eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
sample_file = sorted(ordinary.glob("*.derbake"))[0]
source_rhythm = parse_derbake(sample_file).rhythm
prompt_rhythm = RhythmSequence((source_rhythm.cycles[0],))
review_dir = storage_root / "reviews" / variant
review_dir.mkdir(parents=True, exist_ok=True)
prompt_path = review_dir / "sample_prompt.derbake"
generated_path = review_dir / f"sample_generated_{checkpoint_sha256[:12]}.derbake"
processor.save_derbake(prompt_rhythm, prompt_path)
inputs = processor(derbake_path=prompt_path, return_tensors="pt")
with torch.no_grad():
    generation = model.generate(
        input_ids=inputs["input_ids"].to(device), num_cycles=1,
        max_new_tokens=2048, seed=1337, return_generation_report=True,
    )
generated = processor.decode_generation(generation.continuation_ids[0],
                                        expected_num_cycles=1)
processor.save_derbake(generated, generated_path)
assert parse_derbake(generated_path).rhythm == generated
print("Generation report:", generation.report)
print("Generated file:", generated_path)
print(generated_path.read_text(encoding="utf-8"))
del model


## 10. Upload the approved snapshot to Hugging Face

Set `PUBLISH_TO_HF = True` **only after** reviewing the training metrics and generated rhythm. This uploads the complete validated snapshot to the existing model repo for the selected variant. It does not upload the raw dataset or resumable training checkpoints. Your Hugging Face login must have write access. `upload_folder` creates a new model-repo commit; files already in that repo with the same names are updated. This cell is intentionally disabled by default.

In [ ]:
from huggingface_hub import HfApi

repo_id = {"with_subd": "Appodoom/ldm-with-subd",
           "no_subd": "Appodoom/ldm-no-subd"}[variant]
PUBLISH_TO_HF = False  # Set True only when this trained model is approved for release.
published_commit = None
if not PUBLISH_TO_HF:
    print("Upload skipped. Review metrics and generated rhythm, then set PUBLISH_TO_HF = True.")
else:
    assert metrics["history"][-1]["epoch"] >= target_epochs
    assert release_manifest["source_checkpoint_sha256"] == hashlib.sha256(
        best_checkpoint.read_bytes()).hexdigest()
    subprocess.run([sys.executable, "-m", "ldm.cli", "validate",
                    "--snapshot", str(snapshot)], check=True)
    api = HfApi()
    api.repo_info(repo_id=repo_id, repo_type="model")
    uploaded = api.upload_folder(
        repo_id=repo_id, repo_type="model", folder_path=str(snapshot),
        commit_message=f"Release {variant} model",
    )
    published_commit = uploaded.oid
    print("Uploaded:", f"https://huggingface.co/{repo_id}/tree/{published_commit}")
    print("Commit:", published_commit)


## 11. Verify the uploaded commit and save the release record

The remote check downloads the exact commit returned by the upload, runs the isolated snapshot validation on those downloaded files, and loads the model and processor by repository ID and commit SHA. A successful result records the released revision in Google Drive for reproducible use.

In [ ]:
if published_commit is None:
    print("Remote validation skipped because no upload was made in this session.")
else:
    remote_snapshot = Path(snapshot_download(
        repo_id=repo_id, repo_type="model", revision=published_commit,
        local_dir=project / "remote_validation" / variant / published_commit,
    ))
    subprocess.run([sys.executable, "-m", "ldm.cli", "validate",
                    "--snapshot", str(remote_snapshot)], check=True)
    remote_model = AutoModel.from_pretrained(
        repo_id, revision=published_commit, trust_remote_code=True).eval()
    remote_processor = AutoProcessor.from_pretrained(
        repo_id, revision=published_commit, trust_remote_code=True)
    assert remote_model.config.vocab_hash == remote_processor.tokenizer.atomic.hash
    del remote_model, remote_processor
    release_record = {
        "repo_id": repo_id, "revision": published_commit, "variant": variant,
        "source_code_commit": code_commit,
        "best_checkpoint_sha256": checkpoint_sha256,
        "dataset_fingerprint": metrics["dataset_fingerprint"],
        "best_val_loss": metrics["best_val_loss"],
        "test_loss": metrics["test_loss"],
    }
    record_path = storage_root / "releases" / f"{variant}-{published_commit[:12]}.json"
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_text(json.dumps(release_record, indent=2) + "\n", encoding="utf-8")
    print("Verified remote commit:", published_commit)
    print("Release record:", record_path)


## Use a released model

After remote validation passes, consumers can load `repo_id` with the recorded `revision` and `trust_remote_code=True`. Use the **same revision** for the model and processor. Repeat the notebook separately with `variant = "no_subd"` to train and release the other model.